# Smokies Wildlife Classifier: train on a free GPU

1. **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.
2. About 30–40 minutes the first time (downloading ~10,000 photos, then two training runs).
3. At the end, `smokies_results.zip` is saved to your Google Drive (`MyDrive/smokies/`). Tell Claude when it's there.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
REPO = 'https://github.com/jleonar9/smokies-wildlife-classifier.git'
!git clone -q $REPO /content/proj || (cd /content/proj && git pull -q)
%cd /content/proj
!pip install -q timm

## Photos
The photo list (`data/manifest.csv`) is in the repo. Downloads are cached in Google Drive so a second run skips them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
CACHE = pathlib.Path('/content/drive/MyDrive/smokies'); CACHE.mkdir(parents=True, exist_ok=True)
if (CACHE / 'images.zip').exists():
    !unzip -q -o {CACHE}/images.zip -d /content/proj/data
!python src/download.py --workers 16
if not (CACHE / 'images.zip').exists():
    !cd /content/proj/data && zip -q -r {CACHE}/images.zip images download_report.json

In [ ]:
import json; print(json.dumps(json.load(open('data/download_report.json'))['totals'], indent=2))

## Baseline: linear probe
Freeze the pretrained ConvNeXt-Tiny and train only the final layer. This is the bar fine-tuning has to beat.

In [ ]:
!python src/train.py --run convnext_probe --linear-probe --epochs 6 --lr 1e-3 --head-lr-mult 1 --workers 2

## Full fine-tune
All layers train, with a 10× higher learning rate on the new classifier, label smoothing, TrivialAugment and class-balanced sampling.

In [ ]:
!python src/train.py --run convnext_ft --epochs 12 --workers 2

## Test on photos taken inside the park

In [ ]:
!python src/evaluate.py --runs convnext_probe convnext_ft

In [ ]:
from IPython.display import Image, display
for f in ['confusion_matrix.png', 'errors.png', 'gradcam.png', 'confidence.png']:
    display(Image(f'reports/{f}'))

## Save results to Drive

In [ ]:
!zip -q -r {CACHE}/smokies_results.zip models/convnext_ft models/convnext_probe/config.json models/convnext_probe/history.json reports
!ls -lh {CACHE}